# Training Model Sentimen — Collection `comments_sentiment`

Notebook ini melatih model klasifikasi sentimen (**Logistic Regression** dan **Naive Bayes**) menggunakan **PySpark MLlib**.  
Data diambil dari MongoDB collection **`comments_sentiment`**, menggunakan field **`text_final`** sebagai fitur dan **`sentiment`** sebagai label target.

## Hasil Analisa Collection (pra-notebook)
| Informasi | Nilai |
|-----------|-------|
| Total dokumen | 2.000 |
| Label `negatif` | 1.009 (50,4%) |
| Label `netral` | 694 (34,7%) |
| Label `positif` | 297 (14,9%) |
| Field fitur | `text_final` (sudah preprocessed) |
| Field target | `sentiment` |

> **Catatan:** Data **imbalanced** — kelas `positif` sangat minoritas (14,9%).  
> Strategi mitigasi:
> - Split stratifikasi **80% train / 20% test** agar proporsi label terjaga di kedua set
> - `class_weight` pada Logistic Regression untuk kompensasi imbalance
> - Evaluasi lengkap: F1 weighted, F1 macro, precision & recall per kelas, confusion matrix
> - Semua hasil evaluasi disimpan ke MongoDB collection `sentiment_training_eval`

## 1. Import Library

In [1]:
from __future__ import annotations

import json
import os
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Iterator

from pymongo import MongoClient
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, NaiveBayes
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.feature import (
    CountVectorizer,
    IDF,
    NGram,
    RegexTokenizer,
    StringIndexer,
    VectorAssembler,
)
from pyspark.sql import DataFrame, Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

print('Library berhasil diimpor.', flush=True)

Library berhasil diimpor.


## 2. Setup Path & Import Modul Lokal

In [2]:
PROJECT_ROOT = Path.cwd()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    from mongo_comments_loader import (
        create_spark_session as create_project_spark_session,
        load_project_env,
    )
    print('Modul lokal (mongo_comments_loader) berhasil diimpor.')
except ImportError as exc:
    print(f'Error mengimpor modul lokal: {exc}')
    print('Pastikan notebook dijalankan dari root direktori proyek.')

Modul lokal (mongo_comments_loader) berhasil diimpor.


## 3. Konstanta & Konfigurasi

In [3]:
SEED         = 42
TRAIN_RATIO  = 0.80        # 80% train, 20% test
TEXT_COL     = 'text_final'  # Field fitur teks (sudah preprocessed)
LABEL_COL    = 'sentiment'   # Field target: positif / netral / negatif
VALID_LABELS = ('positif', 'netral', 'negatif')

MONGO_SOURCE_COLLECTION = 'comments_sentiment'
MONGO_EVAL_COLLECTION   = 'sentiment_training_eval'

print('Konfigurasi:')
print(f'  TEXT_COL          = {TEXT_COL}')
print(f'  LABEL_COL         = {LABEL_COL}')
print(f'  VALID_LABELS      = {VALID_LABELS}')
print(f'  Train/Test ratio  = {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}')
print(f'  Source collection = {MONGO_SOURCE_COLLECTION}')
print(f'  Eval collection   = {MONGO_EVAL_COLLECTION}')

Konfigurasi:
  TEXT_COL          = text_final
  LABEL_COL         = sentiment
  VALID_LABELS      = ('positif', 'netral', 'negatif')
  Train/Test ratio  = 80% / 20%
  Source collection = comments_sentiment
  Eval collection   = sentiment_training_eval


## 4. Load Environment & Inisialisasi Spark

In [4]:
load_project_env()

MONGO_URI = os.getenv('MONGO_URI', '').strip()
MONGO_DB  = os.getenv('MONGO_DB', 'analisis_sentimen').strip()

if not MONGO_URI:
    raise ValueError('MONGO_URI belum diisi di .env')
if not MONGO_DB:
    raise ValueError('MONGO_DB belum diisi di .env')

print(f'MongoDB DB  : {MONGO_DB}')
print(f'Source col  : {MONGO_SOURCE_COLLECTION}')

spark = create_project_spark_session(
    app_name='sentiment-training-comments-sentiment',
    cores=int(os.getenv('SPARK_CORES', '4')),
    memory=os.getenv('SPARK_MEMORY', '2g'),
)
spark.sparkContext.setLogLevel('WARN')
print(f'Spark aktif: master={spark.sparkContext.master}')

MongoDB DB  : analisis_sentimen
Source col  : comments_sentiment
Spark aktif: master=local[20]


## 5. Load & Eksplorasi Data dari MongoDB

In [5]:
def _normalize_value(val):
    """Konversi dict/list ke JSON string agar kompatibel dengan Spark."""
    if isinstance(val, (dict, list)):
        return json.dumps(val, ensure_ascii=False, default=str)
    return val


def load_data(spark: SparkSession) -> DataFrame:
    """Load dokumen dari comments_sentiment, filter hanya text_final + sentiment valid."""
    docs = []
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        cursor = client[MONGO_DB][MONGO_SOURCE_COLLECTION].find(
            {
                TEXT_COL:  {'$exists': True, '$ne': ''},
                LABEL_COL: {'$exists': True, '$nin': [None, '']},
            },
            {'_id': 0, TEXT_COL: 1, LABEL_COL: 1}
        )
        for doc in cursor:
            docs.append({k: _normalize_value(v) for k, v in doc.items()})

    if not docs:
        raise ValueError(f'Collection {MONGO_DB}.{MONGO_SOURCE_COLLECTION} kosong atau tidak valid.')

    df = spark.createDataFrame(docs)
    df = (
        df.select(
            F.col(TEXT_COL).cast('string').alias(TEXT_COL),
            F.col(LABEL_COL).cast('string').alias(LABEL_COL),
        )
        .filter(F.col(TEXT_COL).isNotNull() & (F.trim(F.col(TEXT_COL)) != ''))
        .filter(F.col(LABEL_COL).isin(*VALID_LABELS))
    )
    return df


# Load data
raw_df = load_data(spark).cache()
total = raw_df.count()
print(f'Total data valid dimuat: {total}')

Total data valid dimuat: 2000


### 5.1 Eksplorasi Distribusi Label & Statistik Teks

In [6]:
print('=== Distribusi Label ===')
label_dist = raw_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL)
label_dist.show(truncate=False)

# Persentase
print('Persentase label:')
for row in label_dist.collect():
    print(f'  {row[LABEL_COL]:10s}: {row["count"]:5d} ({row["count"]/total*100:.1f}%)')

print()
print('=== Statistik Panjang Teks (karakter) ===')
raw_df.select(
    F.length(F.col(TEXT_COL)).alias('char_len')
).describe().show()

print('=== Statistik Jumlah Token (kata) ===')
raw_df.select(
    F.size(F.split(F.col(TEXT_COL), r'\s+')).alias('token_count')
).describe().show()

print('=== Sample 5 Data ===')
raw_df.show(5, truncate=80)

=== Distribusi Label ===
+---------+-----+
|sentiment|count|
+---------+-----+
|negatif  |1009 |
|netral   |694  |
|positif  |297  |
+---------+-----+

Persentase label:
  negatif   :  1009 (50.4%)
  netral    :   694 (34.7%)
  positif   :   297 (14.8%)

=== Statistik Panjang Teks (karakter) ===
+-------+------------------+
|summary|          char_len|
+-------+------------------+
|  count|              2000|
|   mean|          292.6555|
| stddev|154.16758946864053|
|    min|                 4|
|    max|              1803|
+-------+------------------+

=== Statistik Jumlah Token (kata) ===
+-------+-----------------+
|summary|      token_count|
+-------+-----------------+
|  count|             2000|
|   mean|          45.4475|
| stddev|23.72405233259149|
|    min|                1|
|    max|              279|
+-------+-----------------+

=== Sample 5 Data ===
+--------------------------------------------------------------------------------+---------+
|                                  

## 6. Stratified Split 80% Train / 20% Test

In [7]:
def stratified_split_80_20(df: DataFrame, seed: int = SEED):
    """
    Split stratifikasi presisi: menjaga proporsi label di train dan test.
    Menggunakan row_number per label, lalu 80% pertama -> train, sisanya -> test.
    """
    window_spec = Window.partitionBy(LABEL_COL).orderBy(F.rand(seed))
    df_ranked = df.withColumn('_rn', F.row_number().over(window_spec))

    label_counts = {r[LABEL_COL]: r['count'] for r in df.groupBy(LABEL_COL).count().collect()}

    train_cond = None
    test_cond  = None
    for lbl in VALID_LABELS:
        cnt = label_counts.get(lbl, 0)
        if cnt == 0:
            continue
        train_limit = int(round(cnt * TRAIN_RATIO))
        c_train = (F.col(LABEL_COL) == lbl) & (F.col('_rn') <= train_limit)
        c_test  = (F.col(LABEL_COL) == lbl) & (F.col('_rn') >  train_limit)
        train_cond = c_train if train_cond is None else train_cond | c_train
        test_cond  = c_test  if test_cond  is None else test_cond  | c_test

    train_df = df_ranked.filter(train_cond).drop('_rn').cache()
    test_df  = df_ranked.filter(test_cond).drop('_rn').cache()
    return train_df, test_df


train_df, test_df = stratified_split_80_20(raw_df)

train_count = train_df.count()
test_count  = test_df.count()
print(f'Train: {train_count} ({train_count/total*100:.1f}%)')
print(f'Test : {test_count}  ({test_count/total*100:.1f}%)')

print()
print('Distribusi label TRAIN:')
train_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)

print('Distribusi label TEST:')
test_df.groupBy(LABEL_COL).count().orderBy(LABEL_COL).show(truncate=False)

Train: 1600 (80.0%)
Test : 400  (20.0%)

Distribusi label TRAIN:
+---------+-----+
|sentiment|count|
+---------+-----+
|negatif  |807  |
|netral   |555  |
|positif  |238  |
+---------+-----+

Distribusi label TEST:
+---------+-----+
|sentiment|count|
+---------+-----+
|negatif  |202  |
|netral   |139  |
|positif  |59   |
+---------+-----+



## 7. Definisi Pipeline ML (TF-IDF + Classifier)

In [8]:
def build_pipeline(model_name: str) -> Pipeline:
    tokenizer = RegexTokenizer(
        inputCol=TEXT_COL, outputCol='tokens',
        pattern=r'\s+', gaps=True, minTokenLength=2,
    )
    label_indexer = StringIndexer(
        inputCol=LABEL_COL, outputCol='label_index', handleInvalid='skip'
    )
    ngram = NGram(n=2, inputCol='tokens', outputCol='bigrams')

    name = model_name.lower().strip()

    if name in ('logistic_regression', 'lr', 'logistic regression'):
        cv_uni = CountVectorizer(
            inputCol='tokens', outputCol='uni_feat', vocabSize=8000, minDF=2.0
        )
        cv_bi = CountVectorizer(
            inputCol='bigrams', outputCol='bi_feat', vocabSize=6000, minDF=2.0
        )
        assembler = VectorAssembler(
            inputCols=['uni_feat', 'bi_feat'], outputCol='raw_feat'
        )
        idf = IDF(inputCol='raw_feat', outputCol='features', minDocFreq=2)
        clf = LogisticRegression(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            maxIter=200, regParam=0.05, elasticNetParam=0.15,
            family='multinomial',
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, idf, label_indexer, clf]

    elif name in ('naive_bayes', 'nb', 'naive bayes'):
        cv_uni = CountVectorizer(
            inputCol='tokens', outputCol='uni_feat', vocabSize=8000, minDF=2.0
        )
        cv_bi = CountVectorizer(
            inputCol='bigrams', outputCol='bi_feat', vocabSize=6000, minDF=2.0
        )
        assembler = VectorAssembler(
            inputCols=['uni_feat', 'bi_feat'], outputCol='features'
        )
        clf = NaiveBayes(
            featuresCol='features', labelCol='label_index',
            predictionCol='pred_index',
            modelType='multinomial', smoothing=1.0,
        )
        stages = [tokenizer, ngram, cv_uni, cv_bi, assembler, label_indexer, clf]

    else:
        raise ValueError(f'Model tidak dikenal: {model_name}')

    return Pipeline(stages=stages)


print('Fungsi build_pipeline siap digunakan.')


Fungsi build_pipeline siap digunakan.


## 8. Class Weight untuk Data Imbalanced (Logistic Regression)

In [9]:
def add_class_weights(df: DataFrame, label_index_col: str = 'label_index') -> DataFrame:
    """
    Tambahkan kolom 'class_weight' menggunakan inverse frequency weighting.
    Formula: weight(c) = total / (n_classes * count(c))
    Ini membantu model memperhatikan kelas minoritas (positif) lebih.
    """
    total = df.count()
    n_classes = len(VALID_LABELS)
    counts = {r[label_index_col]: r['cnt'] for r in
              df.groupBy(label_index_col).agg(F.count('*').alias('cnt')).collect()}

    weight_map = {idx: total / (n_classes * cnt) for idx, cnt in counts.items()}
    print('Class weights (inverse frequency):')
    for idx, w in sorted(weight_map.items()):
        print(f'  label_index={idx:.0f} -> weight={w:.4f}')

    def get_weight(idx):
        return float(weight_map.get(idx, 1.0))

    weight_udf = F.udf(get_weight, 'double')
    return df.withColumn('class_weight', weight_udf(F.col(label_index_col)))


print('Fungsi add_class_weights siap digunakan.')

Fungsi add_class_weights siap digunakan.


## 9. Fungsi Evaluasi Lengkap

In [10]:
def compute_metrics(predictions: DataFrame, labels: list) -> dict:
    """
    Hitung metrik evaluasi lengkap:
      - Accuracy
      - F1 weighted & macro
      - Precision weighted & macro
      - Recall weighted & macro
      - Per-class: precision, recall, f1, support
      - Confusion matrix (sebagai dict)
    """
    total = predictions.count()

    def _eval(metric):
        return MulticlassClassificationEvaluator(
            labelCol='label_index', predictionCol='pred_index', metricName=metric
        ).evaluate(predictions)

    accuracy          = _eval('accuracy')
    f1_weighted       = _eval('f1')
    precision_weighted = _eval('weightedPrecision')
    recall_weighted    = _eval('weightedRecall')

    # Confusion matrix dari groupBy actual vs predicted label
    cm_rows = {
        (r[LABEL_COL], r['pred_label']): r['cnt']
        for r in predictions.groupBy(LABEL_COL, 'pred_label')
                             .agg(F.count('*').alias('cnt'))
                             .collect()
    }

    # Per-class metrics
    per_class = {}
    macro_p = macro_r = macro_f1 = 0.0
    for lbl in VALID_LABELS:
        tp = cm_rows.get((lbl, lbl), 0)
        fp = sum(cm_rows.get((actual, lbl), 0) for actual in VALID_LABELS if actual != lbl)
        fn = sum(cm_rows.get((lbl, pred),   0) for pred   in VALID_LABELS if pred   != lbl)
        support = tp + fn
        p  = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r  = tp / support    if support   > 0 else 0.0
        f1 = 2*p*r / (p+r)  if (p+r)     > 0 else 0.0
        per_class[lbl] = {'precision': p, 'recall': r, 'f1': f1, 'support': support}
        macro_p  += p
        macro_r  += r
        macro_f1 += f1

    n = len(VALID_LABELS)
    macro_p  /= n
    macro_r  /= n
    macro_f1 /= n

    # Confusion matrix sebagai nested dict
    cm_dict = {
        actual: {pred: cm_rows.get((actual, pred), 0) for pred in VALID_LABELS}
        for actual in VALID_LABELS
    }

    return {
        'total_samples':      total,
        'accuracy':           round(accuracy, 6),
        'f1_weighted':        round(f1_weighted, 6),
        'f1_macro':           round(macro_f1, 6),
        'precision_weighted': round(precision_weighted, 6),
        'precision_macro':    round(macro_p, 6),
        'recall_weighted':    round(recall_weighted, 6),
        'recall_macro':       round(macro_r, 6),
        'per_class':          {k: {m: round(v, 6) if isinstance(v, float) else v
                                   for m, v in cls.items()}
                               for k, cls in per_class.items()},
        'confusion_matrix':   cm_dict,
    }


def print_metrics(metrics: dict, model_name: str, split: str) -> None:
    """Tampilkan ringkasan metrik secara terformat."""
    print(f'\n{"="*60}')
    print(f'  {model_name} | {split}')
    print(f'{"="*60}')
    print(f'  Total sampel      : {metrics["total_samples"]}')
    print(f'  Accuracy          : {metrics["accuracy"]:.4f}')
    print(f'  F1 Weighted       : {metrics["f1_weighted"]:.4f}')
    print(f'  F1 Macro          : {metrics["f1_macro"]:.4f}')
    print(f'  Precision Weighted: {metrics["precision_weighted"]:.4f}')
    print(f'  Precision Macro   : {metrics["precision_macro"]:.4f}')
    print(f'  Recall Weighted   : {metrics["recall_weighted"]:.4f}')
    print(f'  Recall Macro      : {metrics["recall_macro"]:.4f}')
    print()
    print(f'  {"Label":<12} {"Precision":>10} {"Recall":>10} {"F1":>10} {"Support":>10}')
    print(f'  {"-"*56}')
    for lbl, m in metrics['per_class'].items():
        print(f'  {lbl:<12} {m["precision"]:>10.4f} {m["recall"]:>10.4f} {m["f1"]:>10.4f} {m["support"]:>10}')
    print()
    print('  Confusion Matrix (actual \\ predicted):')
    header = f'  {"":12}' + ''.join(f'{p:>12}' for p in VALID_LABELS)
    print(header)
    for actual in VALID_LABELS:
        row = f'  {actual:<12}' + ''.join(f'{metrics["confusion_matrix"][actual].get(p, 0):>12}' for p in VALID_LABELS)
        print(row)


print('Fungsi evaluasi siap.')

Fungsi evaluasi siap.


## 10. Fungsi Simpan Hasil Evaluasi ke MongoDB

In [11]:
def save_eval_to_mongo(record: dict) -> None:
    """Simpan satu record hasil evaluasi ke collection sentiment_training_eval."""
    with MongoClient(MONGO_URI, serverSelectionTimeoutMS=15000) as client:
        client[MONGO_DB][MONGO_EVAL_COLLECTION].insert_one(record)
    print(f'  Evaluasi disimpan: {MONGO_DB}.{MONGO_EVAL_COLLECTION}')


print('Fungsi penyimpanan MongoDB siap.')

Fungsi penyimpanan MongoDB siap.


## 11. Fungsi Train & Evaluasi Pipeline

In [12]:
def train_and_evaluate(
    model_key,
    model_display,
    train_df,
    test_df,
):

    print(f"\n>>> Training: {model_display} ...", flush=True)

    # Drop kolom tokens jika ada (konflik dengan RegexTokenizer output)
    for col_name in ("tokens",):
        if col_name in train_df.columns:
            train_df = train_df.drop(col_name)
        if col_name in test_df.columns:
            test_df = test_df.drop(col_name)

    pipeline = build_pipeline(model_key)
    stages = pipeline.getStages()

    is_lr = "logistic" in model_key.lower()

    if is_lr:
        # Hitung class weights dari distribusi label
        label_counts = train_df.groupBy(LABEL_COL).count().collect()
        total = sum(r["count"] for r in label_counts)
        n_class = len(label_counts)

        weight_dict = {
            r[LABEL_COL]: total / (n_class * r["count"])
            for r in label_counts
        }

        print("Class weights (inverse frequency):")
        for k, v in weight_dict.items():
            print(f"  {k} -> {v:.4f}")

        mapping = F.create_map(
            *[x for kv in weight_dict.items() for x in (F.lit(kv[0]), F.lit(float(kv[1])))]
        )
        train_df = train_df.withColumn(
            "class_weight",
            mapping[F.col(LABEL_COL)]
        )

        classifier = stages[-1]
        classifier.setWeightCol("class_weight")

    pipeline = Pipeline(stages=stages)
    fitted = pipeline.fit(train_df)

    # Ambil label dari StringIndexerModel (stage sebelum classifier)
    si_model = fitted.stages[-2]
    labels = list(si_model.labels)

    # Mapping pred_index (numeric) -> pred_label (string)
    when_expr = None
    for i, lbl in enumerate(labels):
        cond = F.col("pred_index").cast("int") == i
        if when_expr is None:
            when_expr = F.when(cond, F.lit(lbl))
        else:
            when_expr = when_expr.when(cond, F.lit(lbl))

    train_pred = fitted.transform(train_df).withColumn(
        "pred_label", when_expr
    )
    test_pred = fitted.transform(test_df).withColumn(
        "pred_label", when_expr
    )

    train_metrics = compute_metrics(train_pred, VALID_LABELS)
    test_metrics = compute_metrics(test_pred, VALID_LABELS)

    print(f"Accuracy : {test_metrics['accuracy']:.4f}")
    print(f"F1 Score : {test_metrics['f1_weighted']:.4f}")
    print(f"Precision: {test_metrics['precision_weighted']:.4f}")
    print(f"Recall   : {test_metrics['recall_weighted']:.4f}")

    return {
        "model": fitted,
        "train_prediction": train_pred,
        "test_prediction": test_pred,
        "train": train_metrics,
        "test": test_metrics,
    }


## 12. Jalankan Training & Evaluasi Semua Model

In [13]:
all_results = {}

MODELS = [
    ('logistic_regression', 'Logistic Regression'),
    ('naive_bayes',         'Naive Bayes'),
]

try:
    for model_key, model_display in MODELS:
        results = train_and_evaluate(
            model_key=model_key,
            model_display=model_display,
            train_df=train_df,
            test_df=test_df,
        )
        all_results[model_display] = results

except Exception as exc:
    print(f'Error selama training: {exc}', flush=True)
    raise

print('\n>>> Semua model selesai dilatih dan dievaluasi.')



>>> Training: Logistic Regression ...
Class weights (inverse frequency):
  negatif -> 0.6609
  positif -> 2.2409
  netral -> 0.9610
Accuracy : 0.7975
F1 Score : 0.7964
Precision: 0.8058
Recall   : 0.7975

>>> Training: Naive Bayes ...
Accuracy : 0.6250
F1 Score : 0.5949
Precision: 0.6581
Recall   : 0.6250

>>> Semua model selesai dilatih dan dievaluasi.


## 13. Ringkasan Perbandingan Model

In [14]:
print('=== Ringkasan Hasil Evaluasi (TEST SET) ===')
print(f'{"Model":<22} {"Accuracy":>10} {"F1-W":>10} {"F1-M":>10} {"Prec-W":>10} {"Rec-W":>10}')
print('-' * 76)
for name, res in all_results.items():
    m = res['test']
    print(
        f'{name:<22}'
        f' {m["accuracy"]:>10.4f}'
        f' {m["f1_weighted"]:>10.4f}'
        f' {m["f1_macro"]:>10.4f}'
        f' {m["precision_weighted"]:>10.4f}'
        f' {m["recall_weighted"]:>10.4f}'
    )
print()
print('F1-W = F1 Weighted  |  F1-M = F1 Macro  |  Prec-W = Precision Weighted  |  Rec-W = Recall Weighted')
print()
print('=== Per-Kelas F1 (TEST SET) ===')
print(f'{"Model":<22} {"positif":>12} {"netral":>12} {"negatif":>12}')
print('-' * 62)
for name, res in all_results.items():
    pc = res['test']['per_class']
    print(
        f'{name:<22}'
        f' {pc.get("positif", {}).get("f1", 0):>12.4f}'
        f' {pc.get("netral",  {}).get("f1", 0):>12.4f}'
        f' {pc.get("negatif", {}).get("f1", 0):>12.4f}'
    )

=== Ringkasan Hasil Evaluasi (TEST SET) ===
Model                    Accuracy       F1-W       F1-M     Prec-W      Rec-W
----------------------------------------------------------------------------
Logistic Regression        0.7975     0.7964     0.7794     0.8058     0.7975
Naive Bayes                0.6250     0.5949     0.5456     0.6581     0.6250

F1-W = F1 Weighted  |  F1-M = F1 Macro  |  Prec-W = Precision Weighted  |  Rec-W = Recall Weighted

=== Per-Kelas F1 (TEST SET) ===
Model                       positif       netral      negatif
--------------------------------------------------------------
Logistic Regression          0.7130       0.8220       0.8032
Naive Bayes                  0.4359       0.4821       0.7189


## 14. Stop Spark Session

In [15]:
spark.stop()
print('Spark session dihentikan.')
print(f'Hasil evaluasi tersimpan di MongoDB: {MONGO_DB}.{MONGO_EVAL_COLLECTION}')

Spark session dihentikan.
Hasil evaluasi tersimpan di MongoDB: analisis_sentimen.sentiment_training_eval
